### np.percentile 

In [1]:
import math

def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def flatten(a):
    if not isinstance(a, list):
        return [a]
    result = []
    for elem in a:
        result.extend(flatten(elem))
    return result


def _percentile_1d(data, q):
    if len(data) == 0:
        raise ValueError("cannot do percentile on an empty array")

    if q < 0 or q > 100:
        raise ValueError("percentile q must be in the range [0, 100]")

    x = sorted(data)
    n = len(x)

    if n == 1:
        return float(x[0])

    pos = (q / 100.0) * (n - 1)
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))

    if lo == hi:
        return float(x[lo])

    weight = pos - lo
    return float(x[lo] * (1 - weight) + x[hi] * weight)


def np_percentile(a, q, axis=None):
    """
    Simplified pure-Python np.percentile.
    - axis=None only
    - q can be a single number or list/tuple of numbers
    - linear interpolation
    """
    shape = get_shape(a)

    if axis is not None:
        raise NotImplementedError("Only axis=None is supported in this version")

    data = flatten(a)

    if isinstance(q, (list, tuple)):
        return [_percentile_1d(data, qi) for qi in q]
    else:
        return _percentile_1d(data, q)

### test cases 

In [2]:
print(np_percentile([1, 2, 3, 4, 5], 0))
print(np_percentile([1, 2, 3, 4, 5], 25))
print(np_percentile([1, 2, 3, 4, 5], 50))
print(np_percentile([1, 2, 3, 4, 5], 75))
print(np_percentile([1, 2, 3, 4, 5], 100))
print(np_percentile([10, 20, 30, 40], 50))
print(np_percentile([10, 20, 30, 40], [25, 50, 75]))
print(np_percentile([[1, 2], [3, 4]], 50))
print(np_percentile([[1, 2], [3, 4]], [0, 50, 100]))
print(np_percentile([1], 50)) 

1.0
2.0
3.0
4.0
5.0
25.0
[17.5, 25.0, 32.5]
2.5
[1.0, 2.5, 4.0]
1.0


In [3]:
print(np_percentile([], 50)) 

ValueError: cannot do percentile on an empty array

In [4]:
print(np_percentile([1, 2, 3], -10)) 

ValueError: percentile q must be in the range [0, 100]

In [5]:
print(np_percentile([1, 2, 3], 120)) 

ValueError: percentile q must be in the range [0, 100]

In [6]:
print(np_percentile([[1, 2], [3]], 50)) 

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)

In [7]:
print(np_percentile([1, 2, 3], 50, axis=0)) 

NotImplementedError: Only axis=None is supported in this version